# 28 - Advanced Experiments and Analyses

This notebook contains a series of advanced experiments to evaluate model robustness, calibration, and stability.

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from sklearn.model_selection import GroupShuffleSplit

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline
from src.models.train_balanced_bagging_ensemble import run_balanced_bagging
from src.eval.session_metrics import aggregate_to_session, session_metrics
from src.eval.lood import LOODEvaluator

paths = load_paths()
logger = setup_logger(level="INFO")

# Load data
df_all = load_and_prepare_data()

2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Removed 5750 duplicate flows (7.92%)
2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Metadata columns present for analysis only: ['source_capture_id', 'source_file']
2026-03-30 12:51

## 29. Balanced-Subset Training Experiment

In [2]:
logger.info("Running balanced-subset training experiment...")

df_train = df_all[df_all['split'] == 'train'].copy()
df_val = df_all[df_all['split'] == 'val'].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

# Create a class-balanced subset
n_minority = df_train['label'].sum()

df_majority = df_train[df_train['label'] == 0]
df_minority = df_train[df_train['label'] == 1]

df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=n_minority,
    random_state=42
)

df_train_balanced = pd.concat([df_majority_downsampled, df_minority])

2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Running balanced-subset training experiment...


In [3]:
# Train on full dataset
logger.info("Training on full dataset...")

results_full = run_balanced_bagging(
    df=pd.concat([df_train, df_val, df_test]),
    output_dir=str(paths.artifacts_dir / 'balanced_subset/full')
)

2026-03-30 12:51:23 | INFO | ai-vpn-firewall | Training on full dataset...


In [4]:
# Train on balanced subset
logger.info("Training on balanced subset...")

results_balanced = run_balanced_bagging(
    df=pd.concat([df_train_balanced, df_val, df_test]),
    output_dir=str(paths.artifacts_dir / 'balanced_subset/balanced')
)

2026-03-30 12:52:32 | INFO | ai-vpn-firewall | Training on balanced subset...


In [5]:
# Compare results

auc_full = results_full['isotonic']['test_overall']['session_metrics']['session_roc_auc']
auc_balanced = results_balanced['isotonic']['test_overall']['session_metrics']['session_roc_auc']

logger.info(f"Full dataset session AUC: {auc_full:.4f}")
logger.info(f"Balanced-subset session AUC: {auc_balanced:.4f}")
logger.info(f"Difference (Balanced - Full): {auc_balanced - auc_full:.4f}")

2026-03-30 12:53:29 | INFO | ai-vpn-firewall | Full dataset session AUC: 0.9287
2026-03-30 12:53:29 | INFO | ai-vpn-firewall | Balanced-subset session AUC: 0.9267
2026-03-30 12:53:29 | INFO | ai-vpn-firewall | Difference (Balanced - Full): -0.0020


## 30. Domain-Leave-Two-Out Experiment

In [6]:
logger.info("Running domain-leave-two-out experiment...")

datasets = ["vnat", "iscx", "usbvpn"]
all_results = []

2026-03-30 12:53:29 | INFO | ai-vpn-firewall | Running domain-leave-two-out experiment...


In [7]:
for train_ds in datasets:

    test_ds = [ds for ds in datasets if ds != train_ds]

    logger.info(f"Training on {train_ds}, testing on {test_ds}")

    df_train_fold = df_all[df_all['dataset'] == train_ds].copy()
    df_test_fold = df_all[df_all['dataset'].isin(test_ds)].copy()

    # Carve out a validation split from the training data (group-aware by capture_id)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    groups = df_train_fold['capture_id']
    train_idx, val_idx = next(gss.split(df_train_fold, groups=groups))

    df_train_fold.iloc[train_idx, df_train_fold.columns.get_loc('split')] = 'train'
    df_train_fold.iloc[val_idx, df_train_fold.columns.get_loc('split')] = 'val'
    df_test_fold['split'] = 'test'

    results = run_balanced_bagging(
        df=pd.concat([df_train_fold, df_test_fold]),
        output_dir=str(paths.artifacts_dir / f'leave_two_out/{train_ds}')
    )

    all_results.append(
        results['isotonic']['test_overall']['session_metrics']['session_roc_auc']
    )

2026-03-30 12:53:29 | INFO | ai-vpn-firewall | Training on vnat, testing on ['iscx', 'usbvpn']
2026-03-30 12:53:32 | INFO | ai-vpn-firewall | Training on iscx, testing on ['vnat', 'usbvpn']
2026-03-30 12:54:07 | INFO | ai-vpn-firewall | Training on usbvpn, testing on ['vnat', 'iscx']


In [8]:
mean_auc = np.mean(all_results)

logger.info(f"Mean Leave-Two-Out AUC: {mean_auc:.4f}")

2026-03-30 12:54:39 | INFO | ai-vpn-firewall | Mean Leave-Two-Out AUC: 0.5593


## 31. Calibration Comparison Table

In [9]:
logger.info("Generating calibration comparison table...")

preds_df = pd.read_csv(paths.artifacts_dir / 'balanced_bagging/predictions.csv')

test_preds = preds_df[preds_df['split'] == 'test']

2026-03-30 12:54:39 | INFO | ai-vpn-firewall | Generating calibration comparison table...


In [10]:
calibration_results = []

for cal in ["raw", "platt", "iso"]:

    session_df = aggregate_to_session(
        test_preds,
        prob_col=f"prob_{cal}"
    )

    metrics = session_metrics(
        session_df,
        prob_col=f"prob_{cal}"
    )

    metrics['calibration'] = cal

    calibration_results.append(metrics)

df_calibration = pd.DataFrame(calibration_results)

print(
    df_calibration[
        ['calibration', 'session_roc_auc', 'block_recall_at_zero_fp']
    ].to_string(index=False)
)

calibration  session_roc_auc  block_recall_at_zero_fp
        raw         0.918812                      0.4
      platt         0.908911                      0.4
        iso         0.904950                      0.4


## 32. Stability Bootstrap Analysis

In [11]:
logger.info("Running stability bootstrap analysis...")

n_bootstraps = 100

bootstrap_results = []

2026-03-30 12:54:40 | INFO | ai-vpn-firewall | Running stability bootstrap analysis...


In [12]:
for i in range(n_bootstraps):

    df_sample = resample(
        test_preds,
        replace=True,
        random_state=i
    )

    session_df = aggregate_to_session(
        df_sample,
        prob_col='prob_iso'
    )

    metrics = session_metrics(
        session_df,
        prob_col='prob_iso'
    )

    bootstrap_results.append(metrics)

In [13]:
df_bootstrap = pd.DataFrame(bootstrap_results)

mean_recall = df_bootstrap['block_recall_at_zero_fp'].mean()
std_recall = df_bootstrap['block_recall_at_zero_fp'].std()
zero_fp_freq = (df_bootstrap['block_recall_at_zero_fp'] > 0).mean()

logger.info(f"Mean Recall: {mean_recall:.4f}")
logger.info(f"Std Recall: {std_recall:.4f}")
logger.info(f"Zero-FP Frequency: {zero_fp_freq:.4f}")

2026-03-30 12:54:41 | INFO | ai-vpn-firewall | Mean Recall: 0.4505
2026-03-30 12:54:41 | INFO | ai-vpn-firewall | Std Recall: 0.0941
2026-03-30 12:54:41 | INFO | ai-vpn-firewall | Zero-FP Frequency: 1.0000
